# Part 3: Build the Interactive Dashboard (Tableau) and Deliverables
Step 9 and 11 are Tableau clicks. Step 10 prints the dashboard texts and KPI values to copy in. The last cells check the Tableau files and write `README.md`, `reflection.md` and `lab_proof.md`.

Run notebooks 01 and 02 first.

In [1]:
# Load the clean file and set the Tableau edition used (change if needed)
from pathlib import Path
import pandas as pd

TABLEAU_EDITION = "Tableau Desktop Public Edition"

df = pd.read_csv("evaluation_data_clean.csv")
print("PASS clean file loaded:", len(df), "rows")

PASS clean file loaded: 4489 rows


## Step 9: Create the dashboard layout
1. Click **New Dashboard** (bottom bar). Size: **Fixed size > Generic Desktop (1366 x 768)**.
2. Layout, top to bottom:

| Row | Content |
|---|---|
| 1 | Title and subtitle (Step 10) |
| 2 | Filters: model_version, category, date + "how to use" note |
| 3 | KPI cards: overall average, number of evaluations, date range (Step 10) |
| 4 | Score Distribution (left) · Scores by Category (middle) · Category Comparison (right) |
| 5 | Key takeaways text (Step 10) |

3. Drag the three sheets in. If filters do not appear, click a sheet > dropdown arrow > **Filters** > pick the field. Move them to row 2.
4. Filter action: click the **Category Comparison** sheet on the dashboard > funnel icon **Use as Filter**. Clicking a category bar now filters the other charts.
5. Clean up: **Format > Lines** > Grid Lines **None** on each sheet; one font for the whole workbook (**Format > Workbook**).

## Step 10: Titles, descriptions and context
1. **Dashboard > Show Title**, double-click, paste the title and subtitle printed below.
2. KPI cards are three small sheets:
   - **KPI Average**: drag `score` to **Text** on Marks > Measure **Average**, 1 decimal.
   - **KPI Evaluations**: drag the count field to **Text**.
   - **KPI Date Range**: drag `date` to **Text** > **Minimum**, drag it again > **Maximum**. Set date format to dd MMM yyyy.
   Add them to row 3. They follow the filters because the filters apply to all sheets.
3. Drag a **Text** object onto the dashboard for each chart caption, the how-to note and the key takeaways. Paste from below.

In [2]:
# Context card values (check against the KPI sheets with no filters set)
overall = round(df["score"].mean(), 1)
start = pd.to_datetime(df["date"].min()).strftime("%d %b %Y")
end = pd.to_datetime(df["date"].max()).strftime("%d %b %Y")
print("Overall average score:", overall)
print("Number of evaluations:", len(df))
print("Date range:", start, "to", end)

Overall average score: 55.3
Number of evaluations: 4489
Date range: 26 Aug 2026 to 23 Nov 2026


In [3]:
# Business-language facts for the "so what" (no statistical terms on the dashboard)
avg = df.groupby("category")["score"].mean().round(1).sort_values()
weak, strong = avg.index[0], avg.index[-1]
low_share = round((df[df["category"] == weak]["score"] < 50).mean() * 100)
models = df.groupby("model_version")["score"].mean().round(1)
model_gap = round(models.max() - models.min(), 1)

name = lambda c: c.replace("_", " ").title()
print(f"Weakest: {name(weak)} {avg[weak]} | strongest: {name(strong)} {avg[strong]} | {low_share}% of {name(weak)} below 50")
print(f"Gap between best and worst model: {model_gap} points")

Weakest: Reasoning 40.8 | strongest: Instruction Following 72.2 | 69% of Reasoning below 50
Gap between best and worst model: 2.3 points


In [4]:
# Texts to paste into the dashboard
texts = {
    "Title": "Model Evaluation Score Dashboard",
    "Subtitle": f"Performance by category and model version · {len(df)} evaluations · {start} to {end}",
    "How to use": "Use the filters above to explore specific models, categories or dates. Click a bar in Category Performance Comparison to filter the other charts.",
    "Score Distribution caption": "How many evaluations land in each score range. Scores run from 0 to 100, where 100 is a perfect answer.",
    "Scores by Category caption": "Each dot is one evaluation. The box shows where the middle half of scores sits; the line inside is the typical score.",
    "Category Comparison caption": "Average score per task type, best first. The line marks the overall average.",
    "Key takeaways": (
        f"1. {name(weak)} is the weakest area (average {avg[weak]} of 100); {low_share}% of these answers score below 50. Fix this before scaling use cases that need multi-step logic.\n"
        f"2. {name(strong)} is the strongest area (average {avg[strong]}), ready for wider use.\n"
        f"3. All four models score within {model_gap} points of each other: the type of task drives quality more than the choice of model."
    ),
}
for label, value in texts.items():
    print(f"--- {label} ---\n{value}\n")

--- Title ---
Model Evaluation Score Dashboard

--- Subtitle ---
Performance by category and model version · 4489 evaluations · 26 Aug 2026 to 23 Nov 2026

--- How to use ---
Use the filters above to explore specific models, categories or dates. Click a bar in Category Performance Comparison to filter the other charts.

--- Score Distribution caption ---
How many evaluations land in each score range. Scores run from 0 to 100, where 100 is a perfect answer.

--- Scores by Category caption ---
Each dot is one evaluation. The box shows where the middle half of scores sits; the line inside is the typical score.

--- Category Comparison caption ---
Average score per task type, best first. The line marks the overall average.

--- Key takeaways ---
1. Reasoning is the weakest area (average 40.8 of 100); 69% of these answers score below 50. Fix this before scaling use cases that need multi-step logic.
2. Instruction Following is the strongest area (average 72.2), ready for wider use.
3. All fo

## Step 11: Test and refine
1. Click through every filter value; all charts and KPI cards must update. Reset filters after.
2. Click a bar in Category Comparison; the other charts must filter. Click again to clear.
3. Stakeholder check: can someone read each chart without you? Is the weakest area visible in 5 seconds?
4. Save: **File > Save As** > **Tableau Packaged Workbook (.twbx)** > `evaluation_score_dashboard.twbx` in this folder.
5. Screenshot: **Dashboard > Export Image** > `dashboard_screenshot.png` in this folder (filters cleared).

In [6]:
# Check: Tableau deliverables saved in this folder
for f in ["evaluation_data_clean.csv", "data_source.md", "evaluation_score_dashboard.twbx", "dashboard_screenshot.png"]:
    print(("PASS " if Path(f).exists() else "FAIL missing ") + f)

PASS evaluation_data_clean.csv
PASS data_source.md
PASS evaluation_score_dashboard.twbx
PASS dashboard_screenshot.png


In [7]:
# Deliverable: README.md (file map)
readme = f"""# LAB | Evaluation Dashboard (Ayhan Demir)

**Tool:** {TABLEAU_EDITION}
**Dashboard:** one page for non-technical stakeholders showing where models succeed and fail by evaluation category: score distribution, scores by category, category comparison, three KPI cards, filters for model, category and date, and a click-to-filter action.
**Data:** Option B, generated sample data ({len(df)} evaluations). See `data_source.md`.

Start with `lab_proof.md`.

| File | Purpose |
|---|---|
| `lab_proof.md` | Entry point for the reviewer: workflow, run, evidence, output, limitation |
| `01_data_preparation.ipynb` | Part 1: generate, check and clean the data, write `data_source.md` |
| `02_visualizations.ipynb` | Part 2: Tableau steps 4-8 with reference numbers to check each chart |
| `03_dashboard.ipynb` | Part 3: Tableau steps 9-11, dashboard texts, writes README, reflection, lab_proof |
| `evaluation_data.csv` | Raw generated data |
| `evaluation_data_clean.csv` | Clean data connected to Tableau |
| `evaluation_score_dashboard.twbx` | Tableau packaged workbook |
| `dashboard_screenshot.png` | Screenshot of the final dashboard |
| `data_source.md` | Which data option was used and why |
| `reflection.md` | Communication layer reflection |
"""
Path("README.md").write_text(readme, encoding="utf-8")
print("PASS README.md written")

PASS README.md written


In [8]:
# Deliverable: reflection.md
reflection = f"""# Reflection

I treated the dashboard as the communication layer: it answers "what" (scores by category and model) and "so what" (where to act first), and leaves the "how" in the notebooks. On the dashboard there are no p-values, confidence intervals, standard deviations or test names; the box plot is explained as "where the middle half of scores sits" and the takeaways are written as actions, for example fixing {name(weak)} before scaling multi-step use cases. I included three charts, three KPI cards for scope (average, volume, date range), filters with a one-line how-to, and a click-to-filter action so stakeholders can explore without asking for new charts. I excluded summary tables, raw rows, bin settings and the data generation details, which stay in the notebooks.

Next time I would use real evaluation results from a client project instead of synthetic data, because here the model version is random and the "model choice barely matters" message is a feature of the generator, not a finding. I would also add a target line agreed with the business (for example a minimum acceptable score) so the dashboard shows pass or fail against a goal, not only a ranking.
"""
Path("reflection.md").write_text(reflection, encoding="utf-8")
print("PASS reflection.md written")

PASS reflection.md written


In [9]:
# Deliverable: lab_proof.md (entry point for the reviewer)
rows = "\n".join(f"| {c} | {v} |" for c, v in avg.sort_values(ascending=False).items())
proof = f"""# Lab Proof: Evaluation Dashboard

**Tool:** {TABLEAU_EDITION} · **Data:** Option B, generated ([data_source.md](data_source.md)) · File map: [README.md](README.md)

## 1. Workflow (code and export)
1. [01_data_preparation.ipynb](01_data_preparation.ipynb): runs the lab generator script, checks and cleans the data, saves [evaluation_data_clean.csv](evaluation_data_clean.csv).
2. [02_visualizations.ipynb](02_visualizations.ipynb): Tableau connection and the three charts plus filters, each with reference numbers.
3. [03_dashboard.ipynb](03_dashboard.ipynb): dashboard layout, texts, KPI values, deliverable checks.
4. Tableau export: [evaluation_score_dashboard.twbx](evaluation_score_dashboard.twbx).

## 2. One run with its input
- **Input:** lab generator script, `np.random.seed(42)`, 5 categories, 4 model versions, 90 days, 5 to 15 evaluations per category per day.
- **Run:** `01_data_preparation.ipynb` generated {len(df)} evaluations from {start} to {end}; all checks printed PASS (numeric scores, 5 consistent categories, YYYY-MM-DD dates, no missing values, 0 rows removed).

## 3. Evidence
Average score per category from the notebook, used to check the Category Comparison chart in Tableau (overall average {overall}):

| Category | Average score |
|---|---|
{rows}

Filter test: with only gpt-4 selected, the Tableau averages matched the gpt-4 table in `02_visualizations.ipynb`.

## 4. Final output
![Dashboard](dashboard_screenshot.png)

Key takeaways shown on the dashboard:
{texts["Key takeaways"]}

## 5. Failure or limitation
- **Dates move on every run.** The generator builds dates from `datetime.now()`, so re-running notebook 01 on another day shifts the whole date range while the scores stay the same (the seed fixes only the random numbers). The Tableau date filter and KPI date range then no longer match the screenshot. Fix used: generate once, keep the CSV in the repo, and refresh the Tableau extract only when the data really changes.
- **Synthetic data.** The model version is assigned at random, so the near-equal model scores come from the generator, not from real model behaviour.
- **Reference line gotcha.** Tableau's default average line on the comparison chart averages the five bars ({round(avg.mean(), 1)}), not all evaluations ({overall}); the aggregation must be set to Total.
"""
Path("lab_proof.md").write_text(proof, encoding="utf-8")
print(proof)

# Lab Proof: Evaluation Dashboard

**Tool:** Tableau Desktop Public Edition · **Data:** Option B, generated ([data_source.md](data_source.md)) · File map: [README.md](README.md)

## 1. Workflow (code and export)
1. [01_data_preparation.ipynb](01_data_preparation.ipynb): runs the lab generator script, checks and cleans the data, saves [evaluation_data_clean.csv](evaluation_data_clean.csv).
2. [02_visualizations.ipynb](02_visualizations.ipynb): Tableau connection and the three charts plus filters, each with reference numbers.
3. [03_dashboard.ipynb](03_dashboard.ipynb): dashboard layout, texts, KPI values, deliverable checks.
4. Tableau export: [evaluation_score_dashboard.twbx](evaluation_score_dashboard.twbx).

## 2. One run with its input
- **Input:** lab generator script, `np.random.seed(42)`, 5 categories, 4 model versions, 90 days, 5 to 15 evaluations per category per day.
- **Run:** `01_data_preparation.ipynb` generated 4489 evaluations from 26 Aug 2026 to 23 Nov 2026; all checks p